In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification,  get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset, random_split
import os
import time

import torch
import gc

import glob

import copy # Add this line to import the copy module
import sys

from datetime import datetime
import pytz

from data_loader import allocate_malicious_nodes
from data_loader import generate_topology
from defense import get_high_value_defense_nodes,count_minimum_required_defenders
from main_transformer import run_simulation_transformer
# from data_loader import set_seed # Comment out or remove this line to use the custom set_seed below
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity

from data_loader import set_seed


original_stdout = sys.stdout
original_stderr = sys.stderr
import sys 

class DualLogger(object):
    def __init__(self, file_path):
        self.terminal = sys.stdout
        self.log = open(file_path, "a", encoding='utf-8')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def isatty(self):
        return self.terminal.isatty()

    def close(self):
        if self.log:
            self.log.close()
def deep_clean():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
NUM_CLIENTS = 20
MALICIOUS_RATIO = 0.3
GLOBAL_ROUNDS = 15
# 1. Data & Topology

MODEL_CHECKPOINT = "distilbert-base-uncased"
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
train_ds, test_ds = get_data(
    dataset_name='pubmed',
    tokenizer=TOKENIZER
)
test_loader = DataLoader(test_ds, batch_size=256)
client_datasets = distribute_data(train_ds, NUM_CLIENTS)
#client_datasets = distribute_data_one_class(train_ds, NUM_CLIENTS)
topology_type = 'scale_free'
G = generate_topology(NUM_CLIENTS, topology_type)
neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}
placement_strategy='Topology-Aware'

num_mal = int(NUM_CLIENTS * MALICIOUS_RATIO)

defense_budget  = count_minimum_required_defenders(G)#int(NUM_CLIENTS * 0.2)

bf = 0.5
intensity = 0.02




# ==========================================
# 1. Experiment settup
# ==========================================

NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
bf = 1.5
intensity = 1
norm_factor = 15
placement_strategy = 'Topology-Aware'

import pytz
from datetime import datetime


tz = pytz.timezone('Asia/Shanghai')

MALICIOUS_RATIOS = [0.25]

TOPOLOGY_TYPES = [ 'random_regular','scale_free']
MALICIOUS_RATIOS = [0.3]
DEFENSE_RATIOS = [ 0.2]
SEEDS = [1, 2, 3]
MECHANISMS =  ['MAB']
NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
def_ratio = 0.2
ratio = 0.3
mech = 'MAB'
num_mal = int(NUM_CLIENTS * ratio)
current_defense_budget = int(NUM_CLIENTS * def_ratio)
AUDIT_PROBS = [0.8, 0.9]
AGG_PROBS = [0.8, 0.9]
AGG_THRE = [0.4, 0.5]
SAVE_PATH = ''
for topo_type in TOPOLOGY_TYPES:
    for current_seed in SEEDS:
        for aup in AUDIT_PROBS:
            for ap in AGG_PROBS:
                for at in AGG_THRE:
                    param_str = f"AUP{aup}_AP{ap}_AT{at}"
                    csv_filename = f"final_Transformer_{topo_type}_MR{ratio}_{param_str}_seed{current_seed}.csv"
                    log_filename = f"Log_Transformer_{topo_type}_MR{ratio}_{param_str}_seed{current_seed}.txt"

                    full_save_path = os.path.join(SAVE_PATH, csv_filename)
                    log_full_path = os.path.join(SAVE_PATH, log_filename)


                    all_results = []


                    logger = DualLogger(log_full_path)
                    sys.stdout = logger
                    sys.stderr = logger

                    try:
                        print(f"\n{'='*60}")
                        print(f"⏰ : {time.strftime('%Y-%m-%d %H:%M:%S')}")
                        print(f"📡 : Topo={topo_type}, Mal={ratio}, Def={def_ratio}, Seed={current_seed}")
                        print(f"{'='*60}")

                        set_seed(current_seed)


                        G = generate_topology(NUM_CLIENTS, topo_type)
                        neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}


                        malicious_clients, defense_nodes = allocate_malicious_nodes(
                            G, num_mal, current_defense_budget, topology_type=topo_type, placement= 'Topology-Aware'
                        )

                        theo_intensities = calculate_theoretical_intensity(
                            neighbors, malicious_clients, NUM_CLIENTS, bf, lambda_benign=0.3
                        )

                        start_tick = time.time()
                        start_wall_time = datetime.now(pytz.timezone('Asia/Shanghai')).strftime("%Y-%m-%d %H:%M:%S")

                        ctx = mp.get_context('spawn')


                        with ProcessPoolExecutor(max_workers=1, mp_context=ctx) as executor:

                            future = executor.submit(
                                run_simulation_transformer,
                                current_seed, NUM_CLIENTS, defense_nodes, malicious_clients,
                                G, neighbors, client_datasets, test_ds,
                                mechanism=mech,
                                bf=bf,
                                intensity=intensity,
                                GLOBAL_ROUNDS=GLOBAL_ROUNDS,
                                norm_factor=norm_factor,
                                epochs=1,
                                audit_prob=aup,
                                agg_prob=ap,
                                agg_threshold=at
                            )

                            #
                            _, _, accs, asrs = future.result()
                        end_tick = time.time()
                        duration_sec = round(end_tick - start_tick, 2)

                      
                        result_entry_base = {
                            'seed': current_seed, 'mechanism': mech, 'malicious_ratio': ratio,
                            'defense_ratio': def_ratio, 'topology': topo_type,
                            'duration_sec': duration_sec,'audit_prob': aup,
                            'agg_prob': ap,
                            'agg_thre': at,
                        }

                        for i in range(NUM_CLIENTS):
                            client_row = copy.deepcopy(result_entry_base)
                            client_row.update({
                                'client_id': i, 'final_acc': accs[i], 'final_asr': asrs[i],
                                'node_type': 'MAL' if i in malicious_clients else ('DEF' if i in defense_nodes else 'BEN')
                            })
                            all_results.append(client_row)

                        #
                        pd.DataFrame(all_results).to_csv(full_save_path, index=False)

                      

                        sys.stdout = logger

                    except Exception as e:
                        sys.stdout = original_stdout
                    finally:
                        sys.stdout = original_stdout
                        sys.stderr = original_stderr
                        logger.close()

                    deep_clean()


print(f"\n🎉Experiments compeleted: {SAVE_PATH}")


In [ ]:
#Table 3
import os
import glob
import pandas as pd
# ==========================================
# 
# ==========================================
list_df = []
def fmt(val_mean, val_std):
    m = val_mean * 100
    s = (0.0 if pd.isna(val_std) else val_std) * 100
    return f"{m:05.2f} ({s:05.2f})"
SAVE_PATH = ''
files_sens = glob.glob(os.path.join(SAVE_PATH, "final_Transformer*.csv"))


for f in files_sens:
    temp_df = pd.read_csv(f)
    list_df.append(temp_df)

df_results = pd.concat(list_df, ignore_index=True)

df_benign = df_results[df_results['node_type'] != 'MAL'].copy()

analysis_df = df_benign.groupby(['audit_prob', 'agg_prob', 'agg_thre']).agg({
    'final_acc': 'mean',
    'final_asr': 'mean'
}).reset_index()

print(analysis_df)

# ==========================================
# 
# ==========================================
if list_df:

    trial_means = df_benign.groupby(
        ['topology', 'audit_prob', 'agg_prob', 'agg_thre', 'seed']
    )[['final_acc', 'final_asr']].mean().reset_index()

    stats = trial_means.groupby(
        ['topology', 'audit_prob', 'agg_prob', 'agg_thre']
    )[['final_acc', 'final_asr']].agg(['mean', 'std']).reset_index()

    stats.columns = [
        '_'.join(col).strip('_') if isinstance(col, tuple) else col
        for col in stats.columns.values
    ]

    def generate_latex_rows(df_subset):
        rows = []
        df_subset = df_subset.sort_values(['audit_prob', 'agg_prob', 'agg_thre'])
        for _, row in df_subset.iterrows():
            aup = f"{row['audit_prob']:.1f}"
            ap = f"{row['agg_prob']:.1f}"
            at = f"{row['agg_thre']:.1f}"
            acc = fmt(row['final_acc_mean'], row['final_acc_std'])
            asr = fmt(row['final_asr_mean'], row['final_asr_std'])
            rows.append(f"{aup} & {ap} & {at} & {acc} & {asr} \\\\")

        
            if at == "0.5":
                rows.append(r"\cmidrule(lr){2-5}")
        return "\n".join(rows)

    topologies = stats['topology'].unique()
    table_body = ""
    for topo in topologies:
        topo_label = "Scale-free" if topo == 'scale_free' else "Random-regular"
        subset = stats[stats['topology'] == topo]
        table_body += f"\n\\midrule\n\\multicolumn{{5}}{{l}}{{\\textit{{Topology: {topo_label}}}}} \\\\ \n\\midrule\n"
        table_body += generate_latex_rows(subset)

    latex_final = r"""
\begin{table}[htbp]
    \centering
    \caption{Sensitivity Analysis of MAB Hyperparameters(Transformer on PubMed)}
    \label{tab:mab_sensitivity_split}
    \small
    \begin{tabular}{ccccc}
        \toprule
        \textbf{AUP} & \textbf{AP} & \textbf{AT} & \textbf{ACC (\%)} & \textbf{ASR (\%)} \\
        \midrule
""" + table_body + r"""
        \bottomrule
    \end{tabular}
\end{table}
"""

    latex_final = latex_final.replace(r"\cmidrule(lr){2-5}" + "\n\n\\midrule", r"\midrule")
    latex_final = latex_final.replace(r"\cmidrule(lr){2-5}" + "\n\n\\bottomrule", r"\bottomrule")

    print(latex_final)